In [2]:
import cutlass
import cutlass.cute as cute
import torch
from cutlass.cute.runtime import from_dlpack
import cuda.bindings.driver as cuda

def run_and_verify(kernel_jit, inputs_verify_fn, *torch_tensors, grid, block):
    """通用骨架：编译 → 运行 → 用 torch 验证"""
    cute_tensors = [from_dlpack(t).mark_layout_dynamic() for t in torch_tensors]
    stream = cuda.CUstream(0)
    compiled = cute.compile(kernel_jit, *cute_tensors, stream=stream)
    compiled(*cute_tensors, stream=stream)
    torch.cuda.synchronize()
    inputs_verify_fn()   # 里面写 torch 参考实现 + assert_close


### T1 · elementwise add：`out = a*x + b*y`（热身，30 min）
**描述**：一维 fp32，grid-stride 不要求，一 block 管 256 元素、32 线程。
**接口**：
```python
@cute.kernel
def axpby_kernel(x: cute.Tensor, y: cute.Tensor, out: cute.Tensor,
                 n: cutlass.Int32, a: cutlass.Float32, b: cutlass.Float32):
    ...  # TODO
@cute.jit
def axpby(x: cute.Tensor, y: cute.Tensor, out: cute.Tensor,
          n: cutlass.Int32, a: cutlass.Float32, b: cutlass.Float32,
          stream: cuda.CUstream):
    bidx, _, _ = cute.arch.block_idx()
    tid,  _, _ = cute.arch.thread_idx()
    xt = cute.local_tile(x, (256,), (bidx,))     # TODO: y/out 同理
    # TODO: local_partition → 循环写 out
    # axpby_kernel(...) 不在这里启动——launch 在外层，此 jit 函数直接写循环外逻辑？
    # 不对：jit 函数负责 launch，仿照下面形式自己补全：
    # axpby_kernel(x, y, out, n, a, b).launch(grid=(cute.ceil_div(n, 256), 1, 1),
    #                                         block=(32, 1, 1), stream=stream)
```

- **验收**：`torch.testing.assert_close(out, 2.0*x - 1.5*y)`（取 a=2.0, b=-1.5），n = 4096。
- **思考题**：n = 5000 时会发生什么？（对应 02 篇 ceil/越界——先跑出 bug，再讲清楚。）


In [15]:
import cutlass
import cutlass.cute as cute
import torch
from cutlass.cute.runtime import from_dlpack
import cuda.bindings.driver as cuda


@cute.kernel
def axpby_kernel(x: cute.Tensor, y: cute.Tensor, out: cute.Tensor,
                 n: cutlass.Int32, a: cutlass.Float32, b: cutlass.Float32):
    bidx,_,_ = cute.arch.block_idx()
    tidx,_,_ = cute.arch.thread_idx()
    # n_idx = bidx*block_size*block_tile_size+tid*thread_tile_size--默认对齐先
    
    # 默认256元素一block
    # block_layout = cute.make_layout((256,))
    tile_x = cute.local_tile(x,(256,),(bidx,))
    tile_y = cute.local_tile(y,(256,),(bidx,))
    tile_o = cute.local_tile(out,(256,),(bidx,))

    # 切到线程数据--向量化?
    # thread 0: 0-7
    # thread 1: 8-15
    thread_layout = cute.make_layout((32,),) # row-m 用 1D 分区器表达不出来 stride=(8,)
    t_x = cute.local_partition(tile_x,thread_layout,tidx)
    t_y = cute.local_partition(tile_y,thread_layout,tidx)
    t_o = cute.local_partition(tile_o,thread_layout,tidx)

    for i in cutlass.range(cute.size(t_o)):
        t_o[i] = a * t_x[i] + b * t_y[i]

    # i = 0
    # while(i<8):
    #     t_o[i] = a*t_x[i]+b*t_y[i]
    #     i+=1


@cute.jit
def axpby(x: cute.Tensor, y: cute.Tensor, out: cute.Tensor,
          n: cutlass.Int32, a: cutlass.Float32, b: cutlass.Float32,
          stream: cuda.CUstream):
    axpby_kernel(x, y, out, n, a, b).launch(
        grid=(cute.ceil_div(n, 256), 1, 1),
        block=(32, 1, 1), stream=stream)


def element_add_vf(n: int = 4096, a: float = 2.0, b: float = -1.5):
    assert n % 256 == 0
    x   = torch.randn(n, device='cuda', dtype=torch.float32)
    y   = torch.randn(n, device='cuda', dtype=torch.float32)
    out = torch.zeros(n, device='cuda', dtype=torch.float32)

    xc = from_dlpack(x).mark_layout_dynamic()
    yc = from_dlpack(y).mark_layout_dynamic()
    oc = from_dlpack(out).mark_layout_dynamic()
    stream = cuda.CUstream(0)

    compiled = cute.compile(axpby, xc, yc, oc, n, a, b, stream)
    compiled(xc, yc, oc, n, a, b, stream)
    torch.cuda.synchronize()

    torch.testing.assert_close(out, a * x + b * y, rtol=1e-5, atol=1e-5)
    print(f"T1 axpby passed ✓  (n={n}, a={a}, b={b})")


element_add_vf()


T1 axpby passed ✓  (n=4096, a=2.0, b=-1.5)


### T2 · silu：`out = x * sigmoid(x)`（elementwise + 数学函数）

- **描述**：一维 fp32，框架同 T1。sigmoid 自己用 `1 / (1 + exp(-x))` 写。
- **要求**：n = 8192；验证后 benchmark，2060 目标 ≥ 200 GB/s（读+写合计）。
- **验收**：对 `torch.nn.functional.silu`，atol=1e-5。

In [ ]:
import cutlass
import cutlass.cute as cute
import torch
from cutlass.cute.runtime import from_dlpack
import cuda.bindings.driver as cuda


@cute.kernel
def silu_kernel(x: cute.Tensor, out: cute.Tensor,
                 n: cutlass.Int32):
    bidx,_,_ = cute.arch.block_idx()
    tidx,_,_ = cute.arch.thread_idx()
    
    # 默认256元素一block
    # block_layout = cute.make_layout((256,))
    tile_x = cute.local_tile(x,(256,),(bidx,))
    tile_o = cute.local_tile(out,(256,),(bidx,))

    # 切到线程数据
    thread_layout = cute.make_layout((256,),) # row-m 用 1D 分区器表达不出来 stride=(8,)
    t_x = cute.local_partition(tile_x,thread_layout,tidx)
    t_o = cute.local_partition(tile_o,thread_layout,tidx)

    for i in cutlass.range(cute.size(t_o)):
        tmp = 1/(1+cute.math.exp(-t_x[i]))
        t_o[i]=t_x[i]*tmp



@cute.jit
def silu(x: cute.Tensor, out: cute.Tensor,
          n: cutlass.Int32,
          stream: cuda.CUstream):
    silu_kernel(x, out, n).launch(
        grid=(cute.ceil_div(n, 256), 1, 1),
        block=(256, 1, 1), stream=stream)



def silu_vf(n: int = 8192):
    assert n % 256 == 0
    x   = torch.randn(n, device='cuda', dtype=torch.float32)
    out = torch.zeros(n, device='cuda', dtype=torch.float32)

    xc = from_dlpack(x).mark_layout_dynamic()
    oc = from_dlpack(out).mark_layout_dynamic()
    stream = cuda.CUstream(0)

    # 编译
    compiled = cute.compile(silu, xc, oc, n, stream)
    
    # 1. 验证正确性 (使用小 n)
    compiled(xc, oc, n, stream)
    torch.cuda.synchronize()
    torch.testing.assert_close(out, torch.nn.functional.silu(x), rtol=1e-5, atol=1e-5)
    print(f"T2 silu passed ✓  (n={n})")

    # 2. Benchmark 性能 (使用大 n 来测试真实带宽)
    n_bench = 8192 * 4096  # 约 3355 万个元素，数据量 128MB (读+写合计 256MB)
    x_bench = torch.randn(n_bench, device='cuda', dtype=torch.float32)
    out_bench = torch.zeros(n_bench, device='cuda', dtype=torch.float32)

    xc_bench = from_dlpack(x_bench).mark_layout_dynamic()
    oc_bench = from_dlpack(out_bench).mark_layout_dynamic()
    
    iters = 100
    
    # 预热 (避免首次启动和内存分配影响)
    for _ in range(5):
        compiled(xc_bench, oc_bench, n_bench, stream)
    torch.cuda.synchronize()

    # 使用 CUDA Event 精确计时
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    torch.cuda.synchronize()
    start.record()
    for _ in range(iters):
        compiled(xc_bench, oc_bench, n_bench, stream)
    end.record()
    torch.cuda.synchronize()
    
    # 获取 GPU 执行时间 (毫秒)
    elapsed_time = start.elapsed_time(end) 
    avg_time = elapsed_time / iters / 1000.0  # 转换为秒
    
    # 计算吞吐量
    # 读+写合计: n_bench个fp32(4 bytes)读，n_bench个fp32写
    total_bytes = n_bench * 4 * 2  
    throughput_gbs = (total_bytes / avg_time) / (1024**3)

    print(f"Benchmark: {iters} iters, n={n_bench}")
    print(f"Avg GPU time: {avg_time * 1000:.4f} ms")
    print(f"Throughput: {throughput_gbs:.2f} GB/s (Target >= 200 GB/s)")

silu_vf()


T2 silu passed ✓  (n=8192)
Benchmark: 100 iters, n=33554432
Avg GPU time: 0.3514 ms
Throughput: 711.45 GB/s (Target >= 200 GB/s)


### T3 · warp row-sum（reduction 入门）
- **描述**：矩阵 `x[M, N]`，**每行分配一个 block（1 warp）**，行求和写入 `out[M]`。
- **框架**：
```python
@cute.kernel
def row_sum_kernel(x: cute.Tensor, out: cute.Tensor, M: cutlass.Int32, N: cutlass.Int32):
    bidx, _, _ = cute.arch.block_idx()
    tid,  _, _ = cute.arch.thread_idx()
    # TODO 1: 用 local_tile 取本行
    #   提示：local_tile 对 2D 张量传 tile_shape=(1, N), coord=(bidx, 0)，
    #        得到的 tile 仍是 2D；确认你拿到的是 1D 视角再 partition
    # TODO 2: local_partition((32,), tid)，循环本地累加 → local_sum
    # TODO 3: s = cute.arch.warp_redux_sync(local_sum, kind="add")
    # TODO 4: if tid == 0: out[bidx] = s     （32 线程都有结果，但只写一次）
```
- **要求**：M=128, N=256（保证 32 | N 整除）。先 `assert N % 32 == 0`。
- **验收**：对 `x.sum(dim=-1)`，atol=1e-3（顺序不同精度略差，正常）。
- **思考题**：TODO 4 的 `if tid == 0` 去掉会怎样？（多线程写同一地址，结果“碰巧”对但数据竞争。）

In [20]:
import cutlass
import cutlass.cute as cute
import torch
from cutlass.cute.runtime import from_dlpack
import cuda.bindings.driver as cuda

@cute.kernel
def row_sum_kernel_(x: cute.Tensor, out: cute.Tensor, M: cutlass.Int32, N: cutlass.Int32):
    bidx, _, _ = cute.arch.block_idx()
    tid,  _, _ = cute.arch.thread_idx()
    # 1,用 local_tile 取本行
    # 一个block 1 row,1 wrap
    row_tile = cute.local_tile(x,(N,),(bidx,))
    # 2: local_partition((32,), tid)，循环本地累加 → local_sum
    # per_thread = N//32
    # 若blocksize>32，则错误？可以设置参数32为blocksize
    thread_layout = cute.make_layout((32,))
    t_tile = cute.local_partition(row_tile,thread_layout,tid)

    local_sum = cutlass.Float32(0.0)
    for i in cutlass.range(cute.size(t_tile)):
        local_sum += t_tile[i]
    # 3: wrap 内和
    # 若多wrap处理一行，则通过共享内存通信？
    s = cute.arch.warp_redux_sync(local_sum, kind="add")
    # 4:（32 线程都有结果，但只写一次）
    if tid == 0:
        out[bidx] = s   

@cute.kernel
def row_sum_kernel_1D(x: cute.Tensor, out: cute.Tensor, M: cutlass.Int32, N: cutlass.Int32):
    bidx, _, _ = cute.arch.block_idx()
    tid,  _, _ = cute.arch.thread_idx()
    
    # 1. 用 local_tile 取本行
    # 假设 x 是一维的扁平数组，每个 block 取出长度为 N 的一段（即一行）
    row_tile = cute.local_tile(x, (N,), (bidx,))
    
    # 2. local_partition((32,), tid)，循环本地累加
    thread_layout = cute.make_layout((32,))
    t_tile = cute.local_partition(row_tile, thread_layout, tid)
    
    local_sum = cutlass.Float32(0.0)
    for i in cutlass.range(cute.size(t_tile)):
        local_sum += t_tile[i]
        
    # 3. warp 内和
    # s = cute.arch.warp_redux_sync(local_sum, kind="add")
    # butterfly redue 使用 shuffle_sync 进行树形规约
    # 偏移量从 16 开始，每次减半，直到 1
    offset = 16
    while offset > 0:
        local_sum += cute.arch.shuffle_sync_bfly(local_sum, offset)
        # 只有当前 lane 加上 offset 后仍然在 32 以内，才需要累加
        offset //= 2
    
    # 4. 只有线程 0 写回结果
    if tid == 0:
        out[bidx] = local_sum    
  
@cute.kernel
def row_sum_kernel_2D(x: cute.Tensor, out: cute.Tensor, M: cutlass.Int32, N: cutlass.Int32):
    bidx, _, _ = cute.arch.block_idx()
    tid,  _, _ = cute.arch.thread_idx()
    
    # 1. 用 local_tile 取本行
    # x 为二维张量 (M, N)，tile_shape=(1, N) 表示取 1行×N列，coord=(bidx, 0) 表示取第 bidx 行
    row_tile = cute.local_tile(x, (1, N), (bidx, 0))
    
    # 2. local_partition 将一行的 N 个元素分配给 32 个线程
    thread_layout = cute.make_layout((1, 32))#,stride=(32,1)
    t_tile = cute.local_partition(row_tile, thread_layout, tid)
    
    local_sum = cutlass.Float32(0.0)
    # 遍历每个线程分到的元素，t_tile 是二维 (1, N/32)
    for i in cutlass.range(cute.size(t_tile)):
        local_sum += t_tile[0, i]
        
    # 3. warp 内树形规约
    # 使用 shuffle_sync_ 进行无分支树形规约 (bfly)
    local_sum += cute.arch.shuffle_sync_bfly(local_sum, 16)
    local_sum += cute.arch.shuffle_sync_bfly(local_sum, 8)
    local_sum += cute.arch.shuffle_sync_bfly(local_sum, 4)
    local_sum += cute.arch.shuffle_sync_bfly(local_sum, 2)
    local_sum += cute.arch.shuffle_sync_bfly(local_sum, 1)
    
    # 4. 只有线程 0 写回结果
    if tid == 0:
        out[bidx] = local_sum

@cute.jit
def row_sum_1D(x: cute.Tensor, out: cute.Tensor,
           M: cutlass.Int32, N: cutlass.Int32,
           stream: cuda.CUstream):
    row_sum_kernel_1D(x, out, M, N).launch(
        grid=(M, 1, 1),  # 直接启动 M 个 Block
        block=(32, 1, 1), stream=stream)

@cute.jit
def row_sum_2D(x: cute.Tensor, out: cute.Tensor,
           M: cutlass.Int32, N: cutlass.Int32,
          stream: cuda.CUstream):
    row_sum_kernel_2D(x, out, M, N).launch(
        grid=(M, 1, 1),  # M 个 Block
        block=(32, 1, 1), stream=stream)


def row_sum_vf_1D(M: int = 8192, N: int = 256):
    # 构造 M行 N列的矩阵，flatten 后送入 kernel
    x_2d = torch.randn(M, N, device='cuda', dtype=torch.float32)
    x = x_2d.flatten()
    out = torch.zeros(M, device='cuda', dtype=torch.float32)

    xc = from_dlpack(x).mark_layout_dynamic()
    oc = from_dlpack(out).mark_layout_dynamic()
    stream = cuda.CUstream(0)

    # 编译
    compiled = cute.compile(row_sum_1D, xc, oc, M, N, stream)
    
    # 1. 验证正确性
    compiled(xc, oc, M, N, stream)
    torch.cuda.synchronize()
    # 与 PyTorch 的行求和做对比
    torch.testing.assert_close(out, x_2d.sum(dim=1), rtol=1e-5, atol=1e-5)
    print(f"Row sum passed ✓  (M={M}, N={N})")

    # 2. Benchmark 性能
    iters = 100
    
    # 预热
    for _ in range(5):
        compiled(xc, oc, M, N, stream)
    torch.cuda.synchronize()

    # 使用 CUDA Event 精确计时
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    torch.cuda.synchronize()
    start.record()
    for _ in range(iters):
        compiled(xc, oc, M, N, stream)
    end.record()
    torch.cuda.synchronize()
    
    elapsed_time = start.elapsed_time(end) 
    avg_time = elapsed_time / iters / 1000.0  # 转换为秒
    
    # 计算吞吐量: 读 x (M*N 个 fp32) + 写 out (M 个 fp32)
    total_bytes = (M * N * 4) + (M * 4)  
    throughput_gbs = (total_bytes / avg_time) / (1024**3)

    print(f"Benchmark: {iters} iters, M={M}, N={N}")
    print(f"Avg GPU time: {avg_time * 1000:.4f} ms")
    print(f"Throughput: {throughput_gbs:.2f} GB/s")


def row_sum_vf_2D(M: int = 8192, N: int = 256):
    # 构造 M行 N列的矩阵，直接作为二维张量送入 kernel
    x_2d = torch.randn(M, N, device='cuda', dtype=torch.float32)
    # 不再 flatten
    out = torch.zeros(M, device='cuda', dtype=torch.float32)

    # 传入二维 x_2d
    xc = from_dlpack(x_2d).mark_layout_dynamic()
    oc = from_dlpack(out).mark_layout_dynamic()
    stream = cuda.CUstream(0)

    # 编译
    compiled = cute.compile(row_sum_2D, xc, oc, M, N, stream)
    
    # 1. 验证正确性
    compiled(xc, oc, M, N, stream)
    torch.cuda.synchronize()
    # 与 PyTorch 的行求和做对比
    torch.testing.assert_close(out, x_2d.sum(dim=1), rtol=1e-5, atol=1e-5)
    print(f"Row sum passed ✓  (M={M}, N={N})")

    # 2. Benchmark 性能
    iters = 100
    
    # 预热
    for _ in range(5):
        compiled(xc, oc, M, N, stream)
    torch.cuda.synchronize()

    # 使用 CUDA Event 精确计时
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    torch.cuda.synchronize()
    start.record()
    for _ in range(iters):
        compiled(xc, oc, M, N, stream)
    end.record()
    torch.cuda.synchronize()
    
    elapsed_time = start.elapsed_time(end) 
    avg_time = elapsed_time / iters / 1000.0  # 转换为秒
    
    # 计算吞吐量: 读 x (M*N 个 fp32) + 写 out (M 个 fp32)
    total_bytes = (M * N * 4) + (M * 4)  
    throughput_gbs = (total_bytes / avg_time) / (1024**3)

    print(f"Benchmark: {iters} iters, M={M}, N={N}")
    print(f"Avg GPU time: {avg_time * 1000:.4f} ms")
    print(f"Throughput: {throughput_gbs:.2f} GB/s")

row_sum_vf_1D()

row_sum_vf_2D()

Row sum passed ✓  (M=8192, N=256)
Benchmark: 100 iters, M=8192, N=256
Avg GPU time: 0.0093 ms
Throughput: 840.14 GB/s
Row sum passed ✓  (M=8192, N=256)
Benchmark: 100 iters, M=8192, N=256
Avg GPU time: 0.0092 ms
Throughput: 851.88 GB/s



### T5 · fused softmax（本篇重头戏）
- **描述**：`x[M, N]` 每行 softmax，1 warp 1 行，三遍法：
```
1. 行 max（本线程循环取 max → warp_redux_sync kind="fmax"）
2. 行 sum(exp(x - max))（→ kind="add"）
3. 归一化写回
```
- **框架**：参考 §1.1–1.4 全部 API，结构 = T3 的切分 + 两次 redux + 一次写回循环。
- **要求**：M=128, N=1024，fp32。
- **验收**：对 `torch.softmax(x, dim=-1)`，atol=1e-5。
- **思考题**：
- 为什么减 max？（数值下溢：`exp(889)` fp32 直接 inf）
- 三遍循环读了三遍 `row_pt`，中间结果能存寄存器少读一遍吗？（提示：`cute.make_fragment_like` / 局部变量数组——这是 flash attention online softmax 的前奏）

In [33]:
import cutlass
import cutlass.cute as cute
import torch
from cutlass.cute.runtime import from_dlpack
import cuda.bindings.driver as cuda

@cute.kernel
def fused_softmax_kernel(x: cute.Tensor, out: cute.Tensor,
                         M: cutlass.Int32, N: cutlass.Int32):
    # row-softmax-safe
    # rowsoftmax(A) = exp(x-m)/rowsum(exp(x-m))
    bidx, _, _ = cute.arch.block_idx()
    tid,  _, _ = cute.arch.thread_idx()
    warp_id = tid // 32
    lane_id = tid % 32

    # Block size 为 256，包含 8 个 warp
    num_warps = 8
    
    # 1. 分配共享内存（8 个 fp32 元素，用于 warp 间通信）
    smem_layout = cute.make_layout((num_warps,))
    smem_ptr  = cute.arch.alloc_smem(cutlass.Float32,4*num_warps)
    smem = cute.make_tensor(smem_ptr,smem_layout)

    # 2. 取出本行的 Tile (1, N) 并分配给 256 个线程
    row_tile = cute.local_tile(x, (1, N), (bidx, 0))
    row_tile_o = cute.local_tile(out, (1, N), (bidx, 0))
    
    thread_layout = cute.make_layout((1, 256), stride=(256, 1))
    t_tile = cute.local_partition(row_tile, thread_layout, tid)
    t_tile_o = cute.local_partition(row_tile_o, thread_layout, tid)
    
    # ===== 1. 扫描最大值 =====
    local_max = cutlass.Float32(-1e10)
    for i in cutlass.range(cute.size(t_tile)):
        val = t_tile[0, i]
        local_max = cute.math.max(local_max, val)
        
    # warp 内求最大值 (使用 shuffle_sync_xor 树形规约)
    local_max = cute.math.max(local_max, cute.arch.shuffle_sync_bfly(local_max, 16))
    local_max = cute.math.max(local_max, cute.arch.shuffle_sync_bfly(local_max, 8))
    local_max = cute.math.max(local_max, cute.arch.shuffle_sync_bfly(local_max, 4))
    local_max = cute.math.max(local_max, cute.arch.shuffle_sync_bfly(local_max, 2))
    local_max = cute.math.max(local_max, cute.arch.shuffle_sync_bfly(local_max, 1))

    # 写入 smem
    if lane_id == 0:
        smem[warp_id] = local_max
    cute.arch.sync_threads()
    
    # 读取 smem，求整个 block 的最大值
    for i in cutlass.range(num_warps):
        local_max = cute.math.max(local_max, smem[i])
    cute.arch.sync_threads()

    # ===== 2. 计算 exp 的和 =====
    local_sum = cutlass.Float32(0.0)
    for i in cutlass.range(cute.size(t_tile)):
        local_sum += cute.math.exp(t_tile[0, i] - local_max)
        
    # warp 内求和
    local_sum += cute.arch.shuffle_sync_bfly(local_sum, 16)
    local_sum += cute.arch.shuffle_sync_bfly(local_sum, 8)
    local_sum += cute.arch.shuffle_sync_bfly(local_sum, 4)
    local_sum += cute.arch.shuffle_sync_bfly(local_sum, 2)
    local_sum += cute.arch.shuffle_sync_bfly(local_sum, 1)

    # 写入 smem
    if lane_id == 0:
        smem[warp_id] = local_sum
    cute.arch.sync_threads()
    
    # 读取 smem，求整个 block 的 sum
    local_sum = cutlass.Float32(0.0)  # <-- 必须先清零！
    for i in cutlass.range(num_warps):
        local_sum += smem[i]
    # cute.arch.sync_threads()# 多余

    # ===== 3. 最后计算并写入 =====
    for i in cutlass.range(cute.size(t_tile)):
        t_tile_o[0, i] = cute.math.exp(t_tile[0, i] - local_max) / local_sum


@cute.jit
def fused_softmax(x: cute.Tensor, out: cute.Tensor,
                  M: cutlass.Int32, N: cutlass.Int32,
                  stream: cuda.CUstream):
    # 每 block 处理 1 行，需要 M 个 Block，每个 Block 256 个线程
    fused_softmax_kernel(x, out, M, N).launch(
        grid=(M, 1, 1),
        block=(256, 1, 1),
        stream=stream
    )


def fused_softmax_vf(M: int = 128, N: int = 1024):
    # 构造 M行 N列的矩阵，直接作为二维张量送入 kernel
    x_2d = torch.randn(M, N, device='cuda', dtype=torch.float32)
    out_2d = torch.zeros(M, N, device='cuda', dtype=torch.float32) # 注意 out 也是二维的

    xc = from_dlpack(x_2d).mark_layout_dynamic()
    oc = from_dlpack(out_2d).mark_layout_dynamic()
    stream = cuda.CUstream(0)

    # 编译
    compiled = cute.compile(fused_softmax, xc, oc, M, N, stream)
    
    # 1. 验证正确性
    compiled(xc, oc, M, N, stream)
    torch.cuda.synchronize()
    
    # 与 PyTorch 的 softmax 做对比
    torch.testing.assert_close(out_2d, torch.softmax(x_2d, dim=-1), rtol=1e-5, atol=1e-5)
    print(f"Fused softmax passed ✓  (M={M}, N={N})")

# 运行测试
fused_softmax_vf()


Fused softmax passed ✓  (M=128, N=1024)


- 这个 kernel 对每一行做了 3 次独立的 global memory 读取:max 那一遍读一次 x,sum 那一遍又读一次,最后 write-back 那一遍再读一次;而且 exp(x - max) 被算了两遍(sum 阶段一次,写出阶段又一次)。对于 softmax 这种纯 memory-bound 算子,这是最大的浪费——理想情况下应该只有 1 次读 + 1 次写,GMEM 流量是现在的 1/3。

- 改法很简单,因为每个线程只负责 4 个元素,完全放得进寄存器:
```python
python
# 第一遍:一次性把自己的 4 个元素读进寄存器,同时求 running max
vals = [t_tile[0, i] for i in cutlass.range(4)]   # 只读一次 GMEM
local_max = cutlass.Float32(-1e10)
for i in cutlass.range(4):
    local_max = cute.math.max(local_max, vals[i])
# ... warp+block reduce local_max（同现在）

# 第二遍:在寄存器上算 exp 并直接缓存，不再回读 x
exp_vals = [cutlass.Float32(0.0)] * 4
local_sum = cutlass.Float32(0.0)
for i in cutlass.range(4):
    exp_vals[i] = cute.math.exp(vals[i] - local_max)
    local_sum += exp_vals[i]
# ... warp+block reduce local_sum（同现在）

# 第三遍:直接用缓存的 exp 值除，不再算 exp，也不再读 x
for i in cutlass.range(4):
    t_tile_o[0, i] = exp_vals[i] / local_sum
```
- 这样对 x 只读一次、对 out 只写一次,exp() 也只算一次。


### T7 · benchmark：softmax 带宽
- **描述**：测 T5 的带宽，与 PyTorch `torch.softmax` 对比。
```python
def bench(fn, iters=200):
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - start) / iters * 1e3   # ms
# 带宽 = (读 N*2 + 写 N*2 + 少量) 字节 / 时间
# M=4096, N=1024, fp32:  每次约 4096*1024*4*2 = 33.5 MB
```
- **达标线**：2060 ≥ 250 GB/s、5070 Ti ≥ 600 GB/s。达不到先查三件事：
1. `cute.compile` 是否在循环外（编译不能计时）；
2. block/线程数是否合理（1 warp 管太长的行会限制并行度——M=4096 时 4096 个 block 够了）；
3. 是否意外 fp64（标量没注解导致）。

In [34]:
import time
import torch
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
import cuda.bindings.driver as cuda

# （此处假设你之前的 fused_softmax_kernel 和 fused_softmax 定义已加载）

def bench(fn, iters=200):
    # 预热
    for _ in range(10):
        fn()
    torch.cuda.synchronize()
    
    # 正式测时
    start = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - start) / iters * 1e3  # ms

def bench_softmax():
    M, N = 4096, 1024
    x_2d = torch.randn(M, N, device='cuda', dtype=torch.float32)
    out_2d = torch.zeros(M, N, device='cuda', dtype=torch.float32)

    xc = from_dlpack(x_2d).mark_layout_dynamic()
    oc = from_dlpack(out_2d).mark_layout_dynamic()
    stream = cuda.CUstream(0)

    # 1. 编译必须在循环外
    print("Compiling CUTLASS DSL kernel...")
    compiled = cute.compile(fused_softmax, xc, oc, M, N, stream)
    print("Compilation done.\n")

    # 2. 包装函数
    cute_fn = lambda: compiled(xc, oc, M, N, stream)
    torch_fn = lambda: torch.softmax(x_2d, dim=-1, out=out_2d)

    # 3. 验证正确性
    cute_fn()
    torch.cuda.synchronize()
    torch.testing.assert_close(out_2d, torch.softmax(x_2d, dim=-1), rtol=1e-5, atol=1e-5)
    print("Correctness check passed ✓")

    # 4. Benchmark
    iters = 200
    cute_ms = bench(cute_fn, iters)
    torch_ms = bench(torch_fn, iters)

    # 5. 带宽计算: 读 x + 写 out = 2 * M * N * 4 bytes
    total_bytes = 2 * M * N * 4  # 33554432 Bytes ≈ 32 MB
    cute_gbs = (total_bytes / (cute_ms * 1e-3)) / (1024**3)
    torch_gbs = (total_bytes / (torch_ms * 1e-3)) / (1024**3)

    print(f"\n--- Benchmark (M={M}, N={N}, {iters} iters) ---")
    print(f"CUTLASS DSL Softmax: {cute_ms:.4f} ms | Bandwidth: {cute_gbs:.2f} GB/s")
    print(f"PyTorch Softmax:     {torch_ms:.4f} ms | Bandwidth: {torch_gbs:.2f} GB/s")
    
    # 达标线检查
    target_gbs = 600  # 5070 Ti 达标线
    if cute_gbs >= target_gbs:
        print(f"\n✅ Passed: {cute_gbs:.2f} GB/s >= {target_gbs} GB/s target.")
    else:
        print(f"\n❌ Failed: {cute_gbs:.2f} GB/s < {target_gbs} GB/s target. 需进一步优化访存。")

# 运行性能测试
bench_softmax()


Compiling CUTLASS DSL kernel...
Compilation done.

Correctness check passed ✓

--- Benchmark (M=4096, N=1024, 200 iters) ---
CUTLASS DSL Softmax: 0.0186 ms | Bandwidth: 1682.28 GB/s
PyTorch Softmax:     0.0144 ms | Bandwidth: 2172.93 GB/s

✅ Passed: 1682.28 GB/s >= 600 GB/s target.



### T6 · RMSNorm（综合）
- **描述**：`out = x / sqrt(mean(x²) + eps) * weight`，每行一个 warp。
- **要点**：
1. 先求 `ss = Σ xᵢ²`（本线程累加 → redux add）；
2. `inv = cute.math.rsqrt(ss / N + eps)`，eps 作为 `cutlass.Float32` 参数传入；
3. 写回循环里逐元素乘 `weight_pt[i]`（weight 也要按同样方式 tile + partition）。
**验收**：对手写 torch 参考（或 `torch.nn.RMSNorm`），atol=1e-4。
**结构提示**：这就是 T3 + T1 的叠加，练“一行里两个 partitioned tensor 对齐遍历”。

In [1]:
import cutlass
import cutlass.cute as cute
import torch
import time
from cutlass.cute.runtime import from_dlpack
import cuda.bindings.driver as cuda

@cute.kernel
def RMSNorm_kernel(x: cute.Tensor, w: cute.Tensor, out: cute.Tensor, eps: cutlass.Float32,
                   M: cutlass.Int32, N: cutlass.Int32):
    # RMSNorm: out = x * rsqrt(s^2/N + eps) * w
    bidx, _, _ = cute.arch.block_idx()
    tid,  _, _ = cute.arch.thread_idx()
    warp_id = tid // 32
    lane_id = tid % 32

    num_warps = 8
    
    # 1. 分配共享内存
    smem_layout = cute.make_layout((num_warps,))
    smem_ptr = cute.arch.alloc_smem(cutlass.Float32, 4*num_warps)
    smem = cute.make_tensor(smem_ptr, smem_layout)

    # 2. 取出本行的 x 和 out Tile (1, N)
    # 注意：w 是 1D 张量 (N,)，不能使用 local_tile 取行！
    row_tile_x = cute.local_tile(x, (1, N), (bidx, 0))
    row_tile_o = cute.local_tile(out, (1, N), (bidx, 0))

    # 直接对 w 进行 partition，所有 block 共享同一个 w
    w_layout = cute.make_layout((256,))
    t_tile_w = cute.local_partition(w,w_layout , tid)
    thread_layout = cute.make_layout((1, 256), stride=(256, 1))
    t_tile_x = cute.local_partition(row_tile_x, thread_layout, tid)
    t_tile_o = cute.local_partition(row_tile_o, thread_layout, tid)


    # ===== 1. 分配寄存器 Tensor 并加载 x =====
    # 获取当前线程需要处理的元素个数
    #vec_size = cute.size(t_tile_x)
    #reg_layout = cute.make_layout((vec_size,))
    # 分配位于寄存器中的 Tensor
    #thread_value_x = cute.make_tensor(cute.arch.all, cutlass.Float32, reg_layout)
    
    # 将 x 加载到寄存器中
    # for i in cutlass.range(vec_size):
    #     thread_value_x[i] = t_tile_x[0, i]

    # ===== 2. 计算 x*x 的和 =====
    local_mean_xx = cutlass.Float32(0.0)
    for i in cutlass.range(cute.size(t_tile_x)):
        #local_mean_xx += thread_value_x[i] * thread_value_x[i]
        # 读取 x 到局部变量，底层会被编译器提升为寄存器
        val = t_tile_x[0, i]
        local_mean_xx += val * val
        
    # warp 内求和
    local_mean_xx += cute.arch.shuffle_sync_bfly(local_mean_xx, 16)
    local_mean_xx += cute.arch.shuffle_sync_bfly(local_mean_xx, 8)
    local_mean_xx += cute.arch.shuffle_sync_bfly(local_mean_xx, 4)
    local_mean_xx += cute.arch.shuffle_sync_bfly(local_mean_xx, 2)
    local_mean_xx += cute.arch.shuffle_sync_bfly(local_mean_xx, 1)

    # 写入 smem
    if lane_id == 0:
        smem[warp_id] = local_mean_xx
    cute.arch.sync_threads()
    
    # 读取 smem，求整个 block 的 sum-xx
    local_mean_xx = cutlass.Float32(0.0)
    for i in cutlass.range(num_warps):
        local_mean_xx += smem[i]

    # ===== 3. 最后计算并写入 =====
    # 移到循环外部，只算一次
    inv = cute.math.rsqrt(local_mean_xx / N + eps)
    
    for i in cutlass.range(cute.size(t_tile_x)):
        t_tile_o[0, i] = t_tile_w[i] * inv * t_tile_x[0, i]


@cute.jit
def RMSNorm(x: cute.Tensor, w: cute.Tensor, out: cute.Tensor, eps: cutlass.Float32,
            M: cutlass.Int32, N: cutlass.Int32,
            stream: cuda.CUstream):
    RMSNorm_kernel(x, w, out, eps, M, N).launch(
        grid=(M, 1, 1),
        block=(256, 1, 1),
        stream=stream
    )


def RMSNorm_vf(M: int = 4096, N: int = 1024):
    # 构造 x (M, N) 和 w (N,)
    x_2d = torch.randn(M, N, device='cuda', dtype=torch.float32)
    w_1d = torch.randn(N, device='cuda', dtype=torch.float32)
    out_2d = torch.zeros(M, N, device='cuda', dtype=torch.float32)
    
    eps_val = 1e-5

    # 转 DSL Tensor
    xc = from_dlpack(x_2d).mark_layout_dynamic()
    wc = from_dlpack(w_1d).mark_layout_dynamic()
    oc = from_dlpack(out_2d).mark_layout_dynamic()
    stream = cuda.CUstream(0)

    # 编译
    print("Compiling kernel...")
    compiled = cute.compile(RMSNorm, xc, wc, oc, cutlass.Float32(eps_val), M, N, stream)
    print("Done.")
    
    # 验证正确性
    compiled(xc, wc, oc, cutlass.Float32(eps_val), M, N, stream)
    torch.cuda.synchronize()
    
    # PyTorch 参考实现
    ms = torch.mean(x_2d**2, dim=-1, keepdim=True)
    ref_out = x_2d * torch.rsqrt(ms + eps_val) * w_1d
    
    torch.testing.assert_close(out_2d, ref_out, rtol=1e-5, atol=1e-5)
    print(f"RMSNorm passed ✓  (M={M}, N={N})")

    # --- Benchmark ---
    def bench(fn, iters=200):
        for _ in range(10): fn()  # warmup
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(iters): fn()
        torch.cuda.synchronize()
        return (time.perf_counter() - start) / iters * 1e3

    cute_fn = lambda: compiled(xc, wc, oc, cutlass.Float32(eps_val), M, N, stream)
    torch_fn = lambda: x_2d * torch.rsqrt(torch.mean(x_2d**2, dim=-1, keepdim=True) + eps_val) * w_1d

    cute_ms = bench(cute_fn)
    torch_ms = bench(torch_fn)

    # 带宽计算: 读 x (M*N*4) + 读 w (N*4) + 写 out (M*N*4)
    total_bytes = 2 * M * N * 4 + N * 4
    cute_gbs = (total_bytes / (cute_ms * 1e-3)) / (1024**3)
    torch_gbs = (total_bytes / (torch_ms * 1e-3)) / (1024**3)

    print(f"\n--- Benchmark (M={M}, N={N}) ---")
    print(f"CUTLASS DSL RMSNorm: {cute_ms:.4f} ms | Bandwidth: {cute_gbs:.2f} GB/s")
    print(f"PyTorch RMSNorm:     {torch_ms:.4f} ms | Bandwidth: {torch_gbs:.2f} GB/s")

# 运行测试
RMSNorm_vf()


Compiling kernel...
Done.
RMSNorm passed ✓  (M=4096, N=1024)

--- Benchmark (M=4096, N=1024) ---
CUTLASS DSL RMSNorm: 0.0145 ms | Bandwidth: 2159.31 GB/s
PyTorch RMSNorm:     0.0555 ms | Bandwidth: 563.38 GB/s
